# Weight Space Programming — 2K Google Colab Render

Render the full **17:21** Dracula-dark explainer at **2560×1440 (2K QHD)**.

**Runtime:** `Runtime → Change runtime type → CPU`. Remotion renders in Chromium on the CPU; a GPU does not speed this up.

### What this notebook does
1. Mounts **Google Drive first** (the final MP4 is saved there).
2. Installs Node.js, ffmpeg and Chromium libraries.
3. Clones `lroe/weight-space-programming` (or accepts a ZIP fallback).
4. Installs npm dependencies and Remotion's headless browser.
5. Renders at **2560×1440** with a live percentage / frame counter / ETA.
6. Verifies the MP4 and copies it to Google Drive.

> A full 2K render is long (roughly 1–3 hours depending on the Colab CPU). Keep the tab open. The notebook prints progress continuously.


In [ ]:
#@title 1. Mount Google Drive (required)
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path

DRIVE_OUTPUT_DIR = Path("/content/drive/MyDrive/WeightSpaceProgramming")
DRIVE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Google Drive is ready.")
print("Final renders will be saved to:", DRIVE_OUTPUT_DIR)


In [ ]:
#@title 2. Install Node.js, ffmpeg and Chromium libraries
import subprocess

script = r"""
set -e
export DEBIAN_FRONTEND=noninteractive

NODE_MAJOR=$(node -p 'process.versions.node.split(".")[0]' 2>/dev/null || echo 0)
if [ "$NODE_MAJOR" -lt 20 ]; then
  curl -fsSL https://deb.nodesource.com/setup_22.x | bash -
  apt-get install -y nodejs
fi

if ! command -v ffmpeg >/dev/null 2>&1; then
  apt-get update -qq
  apt-get install -y -qq ffmpeg
fi

apt-get update -qq
for pkg in libnss3 libatk1.0-0 libatk-bridge2.0-0 libcups2 libdrm2 \
           libxkbcommon0 libxcomposite1 libxdamage1 libxfixes3 \
           libxrandr2 libgbm1 libasound2 libpango-1.0-0 libcairo2; do
  apt-get install -y -qq "$pkg" >/dev/null 2>&1 || true
done

echo "node:   $(node -v)"
echo "npm:    $(npm -v)"
echo "ffmpeg: $(ffmpeg -version | head -1)"
"""

subprocess.run(script, shell=True, executable="/bin/bash", check=True)
print("System dependencies ready.")


In [ ]:
#@title 3. Clone the project (ZIP fallback if the repo is private)
import os
import shutil
import subprocess
import zipfile
from pathlib import Path

REPO_URL = "https://github.com/lroe/weight-space-programming.git"  #@param {type:"string"}
PROJECT_DIR = "/content/weight-space-programming"                #@param {type:"string"}
ALLOW_ZIP_FALLBACK = True                                        #@param {type:"boolean"}

PROJECT_DIR = Path(PROJECT_DIR)

def is_project(path: Path) -> bool:
    return (path / "package.json").is_file() and (path / "src" / "Video.tsx").is_file()

if PROJECT_DIR.exists():
    shutil.rmtree(PROJECT_DIR)

cloned = False
try:
    env = os.environ.copy()
    env["GIT_TERMINAL_PROMPT"] = "0"  # fail fast instead of prompting for private-repo credentials
    subprocess.run(
        ["git", "clone", "--depth", "1", REPO_URL, str(PROJECT_DIR)],
        check=True,
        env=env,
    )
    cloned = is_project(PROJECT_DIR)
except Exception as exc:
    print("Git clone failed:", exc)

if not cloned:
    if not ALLOW_ZIP_FALLBACK:
        raise RuntimeError("Could not clone the repository. Upload the project ZIP manually.")
    print("Upload a ZIP of the project folder (exclude node_modules/, out/ and preview/):")
    from google.colab import files
    uploaded = files.upload()
    zips = [name for name in uploaded if name.lower().endswith(".zip")]
    if not zips:
        raise ValueError("No ZIP uploaded.")

    zip_path = Path("/content") / zips[0]
    extract_dir = Path("/content/_weight_space_upload")
    if extract_dir.exists():
        shutil.rmtree(extract_dir)
    extract_dir.mkdir(parents=True)

    with zipfile.ZipFile(zip_path) as zf:
        zf.extractall(extract_dir)

    candidates = [extract_dir] + [p for p in extract_dir.iterdir() if p.is_dir()]
    root = next((p for p in candidates if is_project(p)), None)
    if root is None:
        root = next(
            (p.parent for p in extract_dir.rglob("package.json") if is_project(p.parent)),
            None,
        )
    if root is None:
        raise ValueError("Could not find package.json + src/Video.tsx in the ZIP.")

    if PROJECT_DIR.exists():
        shutil.rmtree(PROJECT_DIR)
    shutil.copytree(root, PROJECT_DIR)
    zip_path.unlink(missing_ok=True)
    shutil.rmtree(extract_dir, ignore_errors=True)

if not is_project(PROJECT_DIR):
    raise ValueError(f"Not a valid project: {PROJECT_DIR}")

audio = PROJECT_DIR / "public" / "audio.wav"
if not audio.is_file():
    raise FileNotFoundError("public/audio.wav is missing from the project.")

print("Project ready:", PROJECT_DIR)
print("Audio:", audio, f"({audio.stat().st_size / (1024**2):.1f} MB)")
print("Files:", ", ".join(sorted(p.name for p in PROJECT_DIR.iterdir() if p.is_file())))


In [ ]:
#@title 4. Install project dependencies + Remotion browser
import os
import subprocess

os.chdir(PROJECT_DIR)

print("Installing npm packages...")
subprocess.run(
    "npm ci --no-audit --no-fund || npm install --no-audit --no-fund",
    shell=True,
    check=True,
)

print("Ensuring Remotion's headless Chromium is available...")
subprocess.run("npx remotion browser ensure", shell=True, check=True)
print("Dependencies ready.")


In [ ]:
#@title 5. Verify the composition
import os
import subprocess

os.chdir(PROJECT_DIR)
subprocess.run("npx remotion compositions src/index.ts", shell=True, check=True)


In [ ]:
#@title 6. Optional: 2K preview still (fast sanity check)
import os
import subprocess
from pathlib import Path
from IPython.display import Image, display

PREVIEW_SECOND = 33  #@param {type:"number"}
PREVIEW_FRAME = int(PREVIEW_SECOND * 30)
PREVIEW_PATH = Path(PROJECT_DIR) / "out" / "preview-2k.png"
PREVIEW_PATH.parent.mkdir(parents=True, exist_ok=True)

cmd = (
    f'npx remotion still AstraWeightSpace "{PREVIEW_PATH}" '
    f'--frame={PREVIEW_FRAME} --scale=1.3333333333 --log=error'
)
print("Running:", cmd)
subprocess.run(cmd, shell=True, cwd=PROJECT_DIR, check=True)

print("Saved:", PREVIEW_PATH)
display(Image(filename=str(PREVIEW_PATH), width=960))


In [ ]:
#@title 7. Render 2K (2560×1440) with live progress
import os
import re
import shutil
import subprocess
import time
from pathlib import Path

OUTPUT_NAME = "astra-weight-space-2k.mp4"  #@param {type:"string"}
CONCURRENCY = 2                             #@param {type:"slider", min:1, max:8, step:1}
CRF = 18                                    #@param {type:"slider", min:14, max:28, step:1}
X264_PRESET = "fast"                        #@param ["ultrafast", "veryfast", "faster", "fast", "medium", "slow", "slower"]

BASE_WIDTH = 1920
BASE_HEIGHT = 1080
RENDER_WIDTH = 2560
RENDER_HEIGHT = 1440
SCALE = RENDER_WIDTH / BASE_WIDTH  # 1.3333333333 -> 2560×1440

cpu_count = os.cpu_count() or 2
CONCURRENCY = max(1, min(CONCURRENCY, cpu_count))

OUTPUT_DIR = Path(PROJECT_DIR) / "out"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_PATH = OUTPUT_DIR / OUTPUT_NAME

print(f"Target: {RENDER_WIDTH}×{RENDER_HEIGHT} (2K QHD)")
print(f"Scale: {SCALE:.10f} from {BASE_WIDTH}×{BASE_HEIGHT}")
print(f"CPU cores: {cpu_count}; concurrency: {CONCURRENCY}")
print(f"Output: {OUTPUT_PATH}\n")

cmd = (
    f'npx remotion render AstraWeightSpace "{OUTPUT_PATH}" '
    f'--scale={SCALE:.10f} '
    f'--concurrency={CONCURRENCY} '
    f'--gl=swangle '
    f'--codec=h264 '
    f'--audio-codec=aac '
    f'--pixel-format=yuv420p '
    f'--crf={CRF} '
    f'--x264-preset={X264_PRESET} '
    f'--timeout=120000 '
    f'--overwrite '
    f'--log=info'
)

print("Running:\n" + cmd + "\n")

ANSI_RE = re.compile(r"\x1b\[[0-9;]*[A-Za-z]")
RENDERED_RE = re.compile(r"Rendered\s+(\d+)\s*/\s*(\d+)")
ETA_RE = re.compile(r"time remaining:\s*([0-9]+m\s*[0-9]+s|[0-9]+s)")

started = time.time()
proc = subprocess.Popen(
    cmd,
    shell=True,
    cwd=PROJECT_DIR,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    env={**os.environ, "PYTHONUNBUFFERED": "1"},
)

buffer = []
last_pct = -1

while True:
    char = proc.stdout.read(1)
    if not char:
        break
    if char in "\r\n":
        line = ANSI_RE.sub("", "".join(buffer)).strip()
        buffer = []
        if not line:
            continue

        match = RENDERED_RE.search(line)
        if match:
            current, total = map(int, match.groups())
            pct = int(current * 100 / total)
            if pct != last_pct:
                eta = ETA_RE.search(line)
                eta_text = f"  ETA {eta.group(1)}" if eta else ""
                print(f"Rendering 2K: {pct:3d}%  [{current:,}/{total:,} frames]{eta_text}", flush=True)
                last_pct = pct
        elif "Rendered" not in line:
            print(line, flush=True)
    else:
        buffer.append(char)

return_code = proc.wait()
elapsed = time.time() - started
print()

if return_code != 0:
    raise RuntimeError(f"Remotion exited with code {return_code}.")

if not OUTPUT_PATH.is_file():
    raise FileNotFoundError(f"Render finished but the file is missing: {OUTPUT_PATH}")

size_mb = OUTPUT_PATH.stat().st_size / (1024**2)
print(f"Render complete in {elapsed/60:.1f} minutes.")
print(f"File: {OUTPUT_PATH} ({size_mb:.1f} MB)")


In [ ]:
#@title 8. Verify the MP4 and copy it to Google Drive
import json
import shutil
import subprocess
from pathlib import Path

OUTPUT_NAME = "astra-weight-space-2k.mp4"  #@param {type:"string"}
output = Path(PROJECT_DIR) / "out" / OUTPUT_NAME

if not output.is_file():
    raise FileNotFoundError(f"Not found: {output}. Run the render cell first.")

probe = json.loads(
    subprocess.check_output(
        [
            "ffprobe", "-v", "error",
            "-select_streams", "v:0",
            "-show_entries", "stream=width,height,codec_name,duration",
            "-of", "json", str(output),
        ],
        text=True,
    )
)
stream = probe["streams"][0]
print("Video:", stream.get("codec_name"), f'{stream.get("width")}×{stream.get("height")}')
print("Duration:", f'{float(stream.get("duration", 0)) / 60:.2f} min')
print("Size:", f"{output.stat().st_size / (1024**2):.1f} MB")

DRIVE_OUTPUT_DIR = Path("/content/drive/MyDrive/WeightSpaceProgramming")
DRIVE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
destination = DRIVE_OUTPUT_DIR / output.name

print("Copying to Google Drive...")
shutil.copy2(output, destination)
print("Saved to Drive:", destination)


## Troubleshooting

- **Google Drive asks for permission:** this is expected; the final MP4 is copied to `MyDrive/WeightSpaceProgramming/`.
- **`git clone` fails:** the repo may be private. Set `ALLOW_ZIP_FALLBACK = True` and upload a ZIP of the project (exclude `node_modules/`, `out/`, and `preview/`).
- **Chrome / browser errors:** re-run cells 2 and 4. `npx remotion browser ensure` downloads the matching headless browser.
- **Render is slow:** 2K is ~1.8× the pixels of 1080p. Keep the Colab tab open. Lower `CONCURRENCY` if RAM is tight; raise it only if the runtime has enough cores.
- **Runtime disconnected:** `/content` is lost, but the MP4 is copied to Drive in cell 8. For very long renders, keep the tab active and use Colab Pro.
- **Progress output:** the render cell prints one line per 1% with frame count and ETA.
